# LightOnOCR-2 Magyar Fine-tuning (v4 - Multi-font)

**Futtatás előtt:** Runtime → Change runtime type → **T4 GPU**

Ez a verzió **több különböző fonttal** generál tanító adatokat a jobb általánosítás érdekében.

In [ ]:
# 1. Telepítés + Fontok letöltése
!pip install -q transformers>=4.45.0 peft datasets accelerate pillow

# Több font letöltése - mind támogatja a magyar karaktereket
!mkdir -p /content/fonts

# Noto Sans család
!wget -q https://github.com/googlefonts/noto-fonts/raw/main/hinted/ttf/NotoSans/NotoSans-Regular.ttf -O /content/fonts/NotoSans-Regular.ttf
!wget -q https://github.com/googlefonts/noto-fonts/raw/main/hinted/ttf/NotoSans/NotoSans-Bold.ttf -O /content/fonts/NotoSans-Bold.ttf
!wget -q https://github.com/googlefonts/noto-fonts/raw/main/hinted/ttf/NotoSans/NotoSans-Italic.ttf -O /content/fonts/NotoSans-Italic.ttf

# Noto Serif
!wget -q https://github.com/googlefonts/noto-fonts/raw/main/hinted/ttf/NotoSerif/NotoSerif-Regular.ttf -O /content/fonts/NotoSerif-Regular.ttf
!wget -q https://github.com/googlefonts/noto-fonts/raw/main/hinted/ttf/NotoSerif/NotoSerif-Bold.ttf -O /content/fonts/NotoSerif-Bold.ttf

# Liberation fonts (Arial/Times alternatívák)
!wget -q https://github.com/liberationfonts/liberation-fonts/raw/main/liberation-fonts-ttf-2.1.5/LiberationSans-Regular.ttf -O /content/fonts/LiberationSans-Regular.ttf
!wget -q https://github.com/liberationfonts/liberation-fonts/raw/main/liberation-fonts-ttf-2.1.5/LiberationSans-Bold.ttf -O /content/fonts/LiberationSans-Bold.ttf
!wget -q https://github.com/liberationfonts/liberation-fonts/raw/main/liberation-fonts-ttf-2.1.5/LiberationSerif-Regular.ttf -O /content/fonts/LiberationSerif-Regular.ttf
!wget -q https://github.com/liberationfonts/liberation-fonts/raw/main/liberation-fonts-ttf-2.1.5/LiberationMono-Regular.ttf -O /content/fonts/LiberationMono-Regular.ttf

# DejaVu
!wget -q https://github.com/dejavu-fonts/dejavu-fonts/raw/master/ttf/DejaVuSans.ttf -O /content/fonts/DejaVuSans.ttf
!wget -q https://github.com/dejavu-fonts/dejavu-fonts/raw/master/ttf/DejaVuSerif.ttf -O /content/fonts/DejaVuSerif.ttf

!ls -la /content/fonts/
print("\n✓ Fontok letöltve")

In [ ]:
# 2. Fontok tesztelése
from PIL import Image, ImageDraw, ImageFont
from pathlib import Path

FONT_DIR = Path("/content/fonts")
TEST_TEXT = "őűŐŰ öüóőúéáűí ÖÜÓŐÚÉÁŰÍ Árvíztűrő"

# Összes font betöltése
FONTS = []
for font_path in sorted(FONT_DIR.glob("*.ttf")):
    try:
        font = ImageFont.truetype(str(font_path), 24)
        FONTS.append((font_path.stem, str(font_path)))
        print(f"✓ {font_path.stem}")
    except Exception as e:
        print(f"✗ {font_path.stem}: {e}")

print(f"\nÖsszesen {len(FONTS)} font betöltve")

# Vizuális teszt
fig_height = len(FONTS) * 35 + 20
test_img = Image.new("RGB", (600, fig_height), "white")
draw = ImageDraw.Draw(test_img)

y = 10
for name, path in FONTS:
    font = ImageFont.truetype(path, 22)
    draw.text((10, y), f"{name}: {TEST_TEXT}", fill="black", font=font)
    y += 35

display(test_img)
print("\n↑ Ellenőrizd, hogy minden sorban látszanak az ékezetek!")

In [ ]:
# 3. Adatgenerálás több fonttal
import json
import random
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont

HUNGARIAN_WORDS = [
    # ő karakterek - gyakori szavak
    "őr", "őriz", "ők", "ősz", "ősi", "őszinte", "őrült", "őröl",
    "erő", "idő", "mező", "tető", "fő", "nő", "bő", "hő", "jő",
    "belső", "külső", "felső", "alsó", "utolsó", "első", "hátsó",
    "költő", "festő", "vezető", "eladó", "vásárló", "igazgató",
    "Győr", "Debrecen", "Székesfehérvár", "Veszprém",
    "börtön", "könyv", "költség", "közel", "között", "mögött",
    "Csatornadíj", "vízdíj", "díj", "díjszabás", "tandíj",
    "dőlt", "dől", "bedől", "kidől", "eldől", "feldől",
    "tükörfúrógép", "árvíztűrő",
    
    # ű karakterek - gyakori szavak
    "űr", "űrlap", "űrhajó", "gyűrű", "tűz", "fűz", "nyűg",
    "gyűjt", "gyűjtemény", "gyűlés", "gyűlölet",
    "tűnik", "fűszer", "fűszeres", "tűző", "fűző",
    "hűtő", "hűvös", "hűség", "hűtlen", "hűsít",
    "szürke", "szűk", "szűr", "szűz", "szűkít",
    "fülke", "küld", "kürt", "süt", "sűrű",
    "működik", "működés", "műszer", "művelet",
    
    # Kevert
    "őrző", "tűző", "fűző", "űző", "kőtörő",
    "halványszürke", "sötétszürke", "világosszürke",
    "fizetendő", "követendő", "teljesítendő", "megoldandó",
    "összeg", "összesen", "összesítés", "összegzés",
    "kedvezmény", "átutalás", "számlaszám", "ügyintéző",
    "adószám", "cégjegyzékszám", "azonosító", "rendszám",
    "határidő", "lejárat", "esedékesség", "érvényesség",
]

SENTENCE_TEMPLATES = [
    "Fizetendő összeg: {amount} Ft",
    "Csatornadíj: {amount} Ft",
    "Vízdíj alapdíj: {amount} Ft",
    "Kedvezmény összege: {amount} Ft",
    "Késedelmi kamat: {amount} Ft",
    "Adószám: {taxid}",
    "Cégjegyzékszám: {companyid}",
    "Fizetési határidő: {date}",
    "Lejárat: {date}",
    "Számla azonosító: {invoice}",
    "Referencia szám: {ref}",
    "Mérőóra állás: {meter}",
    "IBAN: HU{iban}",
    "Tranzakció azonosító: TRX-{trx}",
]

def generate_text():
    lines = []
    
    # Véletlenszerű szavak
    words = random.sample(HUNGARIAN_WORDS, random.randint(6, 10))
    lines.append(" ".join(words))
    
    # Véletlenszerű mondatok
    for _ in range(random.randint(3, 6)):
        template = random.choice(SENTENCE_TEMPLATES)
        text = template.format(
            amount=f"{random.randint(1,99)} {random.randint(100,999):03d}",
            taxid=f"{random.randint(10000000,99999999)}-{random.randint(1,2)}-{random.randint(10,99)}",
            companyid=f"{random.randint(1,9):02d}-{random.randint(1,9):02d}-{random.randint(100000,999999)}",
            date=f"2025.{random.randint(1,12):02d}.{random.randint(1,28):02d}",
            invoice=f"INV-{random.randint(100000,999999)}",
            ref=f"REF-2025-{random.randint(100000,999999):06d}-HU",
            meter=f"{random.randint(0,99999):08d}",
            iban=f"{random.randint(10,99)} {random.randint(1000,9999)} {random.randint(1000,9999)} {random.randint(1000,9999)} {random.randint(1000,9999)} {random.randint(1000,9999)} {random.randint(1000,9999)}",
            trx=f"2025-{random.randint(1000000,9999999)}",
        )
        lines.append(text)
    
    # Mindig tartalmazza a kritikus tesztsort
    lines.append("öüóőúéáűí - ÖÜÓŐÚÉÁŰÍ")
    
    # Árvíztűrő sor
    lines.append("Árvíztűrő tükörfúrógép - ÁRVÍZTŰRŐ TÜKÖRFÚRÓGÉP")
    
    return "\n".join(lines)

def render_text(text, font_path, width=800, font_size=24):
    font = ImageFont.truetype(font_path, font_size)
    lines = text.split("\n")
    line_height = font_size + 14
    height = len(lines) * line_height + 70
    
    # Véletlenszerű háttér és szövegszín
    bg_colors = ["white", "#fafafa", "#f5f5f5", "#fffef0", "#f0f8ff", "#fff5ee"]
    text_colors = ["black", "#111111", "#1a1a1a", "#222222", "#333333"]
    
    bg_color = random.choice(bg_colors)
    text_color = random.choice(text_colors)
    
    img = Image.new("RGB", (width, height), bg_color)
    draw = ImageDraw.Draw(img)
    
    y = 35
    for line in lines:
        draw.text((35, y), line, fill=text_color, font=font)
        y += line_height
    return img

# Generálás
Path("training_data/images").mkdir(parents=True, exist_ok=True)
annotations = []

NUM_SAMPLES = 500  # Több minta a jobb tanuláshoz
font_usage = {name: 0 for name, _ in FONTS}

for i in range(NUM_SAMPLES):
    text = generate_text()
    
    # Véletlenszerű font kiválasztása
    font_name, font_path = random.choice(FONTS)
    font_usage[font_name] += 1
    
    # Véletlenszerű betűméret
    font_size = random.choice([18, 20, 22, 24, 26, 28, 30])
    
    img = render_text(text, font_path, font_size=font_size)
    img.save(f"training_data/images/{i:05d}.png")
    
    annotations.append({
        "image": f"{i:05d}.png",
        "text": text,
        "font": font_name,
        "font_size": font_size,
    })
    
    if (i+1) % 100 == 0:
        print(f"  {i+1}/{NUM_SAMPLES}")

with open("training_data/annotations.jsonl", "w", encoding="utf-8") as f:
    for a in annotations:
        f.write(json.dumps(a, ensure_ascii=False) + "\n")

print(f"\n✓ Generálva: {NUM_SAMPLES} kép")
print("\nFont használat:")
for name, count in sorted(font_usage.items(), key=lambda x: -x[1]):
    print(f"  {name}: {count}")

# Példák megjelenítése
print("\nPélda képek különböző fontokkal:")
for idx in [0, 50, 100, 150]:
    print(f"\n--- {annotations[idx]['font']} ({annotations[idx]['font_size']}pt) ---")
    display(Image.open(f"training_data/images/{idx:05d}.png"))

In [ ]:
# 4. Modell betöltése
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText

MODEL_ID = "lightonai/LightOnOCR-2-1B-base"

print(f"Modell betöltése: {MODEL_ID}")
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
processor = AutoProcessor.from_pretrained(MODEL_ID)

print(f"✓ Modell betöltve")
print(f"  Device: {model.device}")

In [ ]:
# 5. LoRA konfiguráció - nagyobb rank a jobb tanuláshoz
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,  # Nagyobb rank
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# 6. Dataset
import json
from datasets import Dataset
from PIL import Image

def load_data():
    data = []
    with open("training_data/annotations.jsonl", encoding="utf-8") as f:
        for line in f:
            entry = json.loads(line)
            data.append({
                "image_path": f"training_data/images/{entry['image']}",
                "text": entry["text"]
            })
    return Dataset.from_list(data)

def process_example(example):
    image = Image.open(example["image_path"]).convert("RGB")
    text = example["text"]
    
    image_inputs = processor.image_processor(image, return_tensors="pt")
    text_inputs = processor.tokenizer(
        text,
        return_tensors="pt",
        padding="max_length",
        max_length=512,
        truncation=True,
    )
    
    return {
        "pixel_values": image_inputs["pixel_values"].squeeze(0),
        "input_ids": text_inputs["input_ids"].squeeze(0),
        "attention_mask": text_inputs["attention_mask"].squeeze(0),
        "labels": text_inputs["input_ids"].squeeze(0),
    }

dataset = load_data()
processed_dataset = dataset.map(process_example, remove_columns=dataset.column_names)
print(f"✓ Dataset: {len(processed_dataset)} példa")

In [ ]:
# 7. Training - több epoch
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./lighton-hun-lora",
    num_train_epochs=5,  # Több epoch
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=5e-5,
    warmup_ratio=0.1,
    logging_steps=20,
    save_steps=100,
    save_total_limit=2,
    bf16=True,
    remove_unused_columns=False,
    dataloader_pin_memory=False,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=processed_dataset,
)

print("Tanítás indítása (5 epoch, 500 kép, 11 font)...")
print("Ez ~15-20 percet vesz igénybe T4 GPU-n.")
trainer.train()
print("\n✓ Tanítás kész!")

In [ ]:
# 8. Mentés
print("LoRA adapter mentése...")
model.save_pretrained("./lighton-hun-lora")

print("LoRA súlyok összefésülése...")
merged_model = model.merge_and_unload()
merged_model.save_pretrained("./lighton-hun-merged")
processor.save_pretrained("./lighton-hun-merged")

print("✓ Modell mentve: ./lighton-hun-merged")

In [ ]:
# 9. Teszt több képpel
print("Tesztelés...\n")

test_indices = [0, 100, 200, 300, 400]

for idx in test_indices:
    test_img = Image.open(f"training_data/images/{idx:05d}.png")
    
    inputs = processor.image_processor(test_img, return_tensors="pt")
    inputs = {k: v.to(merged_model.device) for k, v in inputs.items()}
    inputs["input_ids"] = processor.tokenizer("", return_tensors="pt")["input_ids"].to(merged_model.device)
    
    with torch.no_grad():
        outputs = merged_model.generate(**inputs, max_new_tokens=400, do_sample=False)
    
    result = processor.tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    print(f"=== Kép #{idx} ===")
    display(test_img)
    print(f"OCR eredmény:")
    print(result[:500])
    print("\n")

In [ ]:
# 10. Letöltés
!zip -r lighton-hun-merged.zip lighton-hun-merged/

from google.colab import files
files.download("lighton-hun-merged.zip")

print("\n" + "="*60)
print("KÖVETKEZŐ LÉPÉS MAC-EN:")
print("="*60)
print("")
print("1. Kicsomagolás:")
print("   unzip lighton-hun-merged.zip")
print("")
print("2. MLX konverzió:")
print("   mlx_vlm convert --hf-path lighton-hun-merged \\")
print("       --mlx-path models/lighton-hun-mlx -q --q-bits 4")
print("")
print("3. Teszt:")
print("   python run_ocr.py test.pdf --engine lighton \\")
print("       --model models/lighton-hun-mlx --format plain")